In [6]:
import pandas as pd
import numpy as np
from optbinning import BinningProcess, Scorecard
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

df = pd.read_csv("credit_risk_dataset.csv")
df['LTI'] = np.round(df['loan_amnt'] / df['person_income'], 2)
df['loan_int_rate'] = df['loan_int_rate'].fillna(0)
df['person_emp_length'] = df['person_emp_length'].fillna(0)



In [7]:
features = ['LTI', 'loan_grade', 'person_income', 'loan_int_rate',
            'loan_percent_income', 'person_home_ownership', 'loan_intent',
            'person_emp_length', 'person_age', 'loan_amnt']


categorical_cols = ['loan_grade', 'person_home_ownership', 'loan_intent']

x = df[features]
y = df['loan_status']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=67)

In [8]:
#Discretizes continuous variables into categorized bins and transforms them based on their relationship with a target variable
binning_process = BinningProcess(
    variable_names=features,
    categorical_variables=categorical_cols
)

In [12]:
#Score card
lr = LogisticRegression(max_iter=1000)

scorecard = Scorecard(
    binning_process=binning_process,
    estimator=lr,
    scaling_method="pdo_odds",
    scaling_method_params={"pdo": 20, "odds": 1, "scorecard_points": 600}
)

scorecard.fit(x_train, y_train)
print("Scorecard trained")

scorecard_table = scorecard.table(style="detailed")
print(scorecard_table)

scores = scorecard.score(x_test)
print(scores[:10])

df['credit_score'] = scorecard.score(x)
df[['loan_amnt', 'person_income', 'LTI', 'loan_status', 'credit_score']].head(10)

Scorecard trained
     Variable  Bin id                   Bin  Count  Count (%)  Non-event  \
0         LTI       0          (-inf, 0.05)   2771   0.106315       2495   
1         LTI       1          [0.05, 0.07)   2054   0.078806       1822   
2         LTI       2          [0.07, 0.13)   6909   0.265078       6037   
3         LTI       3          [0.13, 0.16)   2892   0.110958       2474   
4         LTI       4          [0.16, 0.19)   1779   0.068255       1455   
..        ...     ...                   ...    ...        ...        ...   
8   loan_amnt       8  [14450.00, 18062.50)   2505   0.096110       1823   
9   loan_amnt       9  [18062.50, 22225.00)   1320   0.050645        891   
10  loan_amnt      10       [22225.00, inf)   1431   0.054903        920   
11  loan_amnt      11               Special      0   0.000000          0   
12  loan_amnt      12               Missing      0   0.000000          0   

    Event  Event rate       WoE        IV        JS  Coefficient     

,loan_amnt,person_income,LTI,loan_status,credit_score
0,35000,59000,0.59,1,501.593247
1,1000,9600,0.10,0,672.724407
2,5500,9600,0.57,1,540.408251
3,35000,65500,0.53,1,552.059358
4,35000,54400,0.64,1,549.599278
5,2500,9900,0.25,1,675.912843
6,35000,77100,0.45,1,593.606616
7,35000,78956,0.44,1,570.422256
8,35000,83000,0.42,1,618.276145
9,1600,10000,0.16,1,599.137907
